# 4a - Goal Seek in Python

**Time:** 45 minutes  |  **AI: explains** (Gemini may explain an error or a line to you)

**What you will do**
- Find the IRR of a project four ways: try rates, halve an interval by hand, `brentq`, and `npf.irr`.
- Meet the one new idea of Session 4: **a function can be an input to another function**.
- Read a `ValueError`, then see why a project can have two IRRs.
- For each cell: write your prediction first, then run it, then compare.

How to work: click a cell, press Shift+Enter to run it. Run the cells **one at a time, in order**. Cell 8 fails on purpose, so do not use "Run all".

## Setup - Install numpy-financial

Colab does not have the package `numpy-financial`. `!pip install` adds a package to this Colab session. The `!` means: run this as a terminal command, not as Python. `-q` keeps the output short.

Run this cell once. It takes a few seconds.

In [ ]:
!pip install -q numpy-financial

## Catch the error (AI off, in pairs)

Two short pieces of code from real finance work, the same as on the slides. Each one runs without an error message, and each one gives a wrong answer. Run it, look at the result, find the problem, then fix it in the cell below it.

### Catch the error 1 - A mine. Hurdle rate 8%. Invest or not?

Open the mine today, earn 16,000 a year for four years, then pay to close it and clean up. `npf.irr` is Excel's IRR (you installed it just above). Run the cell.

In [ ]:
import numpy_financial as npf

mine = [-17000,                # open the mine
        16000, 16000, 16000, 16000,
        -52000]                # close and clean up
npf.irr(mine)

6.77% is below the 8% hurdle, so: reject? Check it two other ways, in the cell below:
1. The NPV at 8%: `npf.npv(0.08, mine)` (it treats the first cash flow as today, unlike Excel's NPV).
2. A colleague's solver found another IRR: 65.4% (65.36%, more precisely). Put it back in: what is the NPV at 0.6536?

Check: the NPV at 8% is **+604**.

In [ ]:
# YOUR CODE: the NPV at 8%

# YOUR CODE: the NPV at 65.36%


**Your answer:** how many IRRs does the mine have? Invest or not, and why? Write it here: ...

### Catch the error 2 - A 10-year lease. 1.1%, 13.1% or 13.9%?

You pay 100,000 today and receive 1,500 every month for 10 years. Run the cell.

In [ ]:
lease = [-100000] + [1500] * 120   # pay today, receive 1,500 a month
npf.irr(lease)

Analyst 1: "1.09%. Below our 8% hurdle. Reject." Analyst 2: "1.09% x 12 = 13.1% a year." Who is right? Each cash flow is one month apart, so the IRR is a rate per month. Turn it into a rate per year in the cell below. Check: you should get **13.9%**.

In [ ]:
irr_month = npf.irr(lease)
# YOUR CODE: the rate per year


## The story

The MD asks: **"What return does this project make?"** The answer is the **IRR**: the discount rate where the NPV is zero. Excel's Goal Seek finds it: you give the target (NPV = 0), Excel finds the input (the rate).

Today you do Goal Seek in Python. The project is the one from Lecture 4: you pay **5,000** today and receive 1,000, 1,000, 2,000 and 2,000 in the next four years.

The code is mostly written. You read it, predict the result, run it, and compare. In 4b you build a solver yourself.

## Cell 1 - NPV at 5% and at 10%

`discount(cash_flows, r)` is the function from Session 1: it adds up the present value of a list of cash flows. It counts the first item as **year 1**. Our list starts at **year 0** (the 5,000 you pay today), which is not discounted. So we split it:

- `cash_flows[0]` is the year-0 amount.
- `cash_flows[1:]` is a **slice**: the list without the first item (years 1 to 4).

NPV = year 0 + `discount` of the rest.

You should get **+232.49** at 5% and **-395.81** at 10%.

**Your prediction:** the NPV is positive at 5% and negative at 10%. Between 5% and 10% it crosses zero. Where is the IRR, roughly?

In [ ]:
cash_flows = [-5000, 1000, 1000, 2000, 2000]   # year 0, 1, 2, 3, 4


def discount(cash_flows, r):
    total = 0
    for t, cf in enumerate(cash_flows, start=1):   # the first item is year 1
        total = total + cf / (1 + r) ** t
    return total


npv_5 = cash_flows[0] + discount(cash_flows[1:], 0.05)
npv_10 = cash_flows[0] + discount(cash_flows[1:], 0.10)

print(f"NPV at 5%:  {npv_5:+,.2f}")
print(f"NPV at 10%: {npv_10:+,.2f}")

## Cell 2 - Try every rate from 0% to 15%

Typing the NPV formula again and again is slow. So we write the question once, as a function: `npv_at(r)` gives the NPV at the rate `r`. It has one input only, the rate.

Then a `for` loop tries every whole percent from 0 to 15. `pct / 100` turns 7 into 0.07.

You should see the NPV change sign once.

**Your prediction:** between which two whole percents does the NPV change from positive to negative?

In [ ]:
def npv_at(r):
    return cash_flows[0] + discount(cash_flows[1:], r)


print(" rate        NPV")
for pct in range(0, 16):
    r = pct / 100
    print(f"{pct:>4}%  {npv_at(r):>9,.2f}")

## Cell 3 - Goal Seek by hand: halve the interval

This is called **bisection**. Start with an interval, 0% to 100%. Look at the middle.

- If the NPV at the middle is still positive, the IRR is higher: move the left end (`low`) up to the middle.
- If it is negative, the IRR is lower: move the right end (`high`) down to the middle.

Each turn halves the interval. A `for` loop repeats it 30 times. After 30 halvings the interval is about one billionth wide.

You should get **6.74%**.

**Your prediction:** after only 1 halving, what is the middle? After 3? (Start at 0 and 1.)

In [ ]:
low, high = 0.0, 1.0          # between 0% and 100%

for step in range(30):
    mid = (low + high) / 2
    if npv_at(mid) > 0:      # still positive: the IRR is higher
        low = mid
    else:                    # negative: the IRR is lower
        high = mid

print(f"IRR after 30 halvings: {mid:.4%}")

## Cell 4 - The solver: `brentq`

Bisection is so common that SciPy has a solver for it: `brentq`. A solver finds the input that makes a function equal to zero.

`from scipy.optimize import brentq` is an **import from a module**: you take one tool from a toolbox.

Look at the call: `brentq(npv_at, 0, 1)`.

- **`npv_at` has no brackets after it.** This is **a function as an input**. You do not run `npv_at` yourself. You hand the question over, and `brentq` asks it many times, with different rates.
- **`0, 1` is the bracket**: search between 0% and 100%.

You should get **6.74%**, the same as the loop.

**Your prediction:** will `brentq` give exactly the same number as your 30 halvings?

(Ask Gemini: *"Explain this line to a finance student: `irr = brentq(npv_at, 0, 1)`"*. Does it say the same as the text above?)

In [ ]:
from scipy.optimize import brentq

irr = brentq(npv_at, 0, 1)    # npv_at: no brackets, we hand over the function

print(f"IRR from brentq: {irr:.4%}")

## Cell 5 - Put the answer back in

Checks card, entry 1: **put the answer back in.** If `irr` is the IRR, the NPV at `irr` must be 0.

The `if` below is a **guard**: if the NPV is not close to zero, `raise ValueError(...)` stops the code with your own message. A solver that fails silently is dangerous.

**Your prediction:** exactly 0?

In [ ]:
npv_at_irr = npv_at(irr)
print(f"NPV at the IRR: {npv_at_irr:.13f}")

if abs(npv_at_irr) > 1e-6:
    raise ValueError("The NPV at the IRR is not zero: the solver failed")

## Cell 6 - A second opinion: `npf.irr`

The package `numpy-financial` has the same functions as Excel's finance functions. You installed it in the Setup cell. `import numpy_financial as npf` loads it and calls it `npf`.

`npf.irr(cash_flows)` takes the whole list, year 0 included. You do not hand it a function: it builds the NPV itself.

You should get **6.74%**.

**Your prediction:** will it agree with `brentq`?

In [ ]:
import numpy_financial as npf

print(f"npf.irr:  {npf.irr(cash_flows):.4%}")
print(f"brentq:   {irr:.4%}")

## Cell 7 - Session 0 slide 10, now readable

On Session 0 slide 10, an AI wrote code to compute the IRR of an equity deal: pay **100** today, receive **250** at the exit, **5 years** later. The AI said **25.7%**. Right answer: **20.1%**. Now you can read why.

`np.zeros(5)` makes a list-like row of five zeros. `cf[0] = -100` sets the first slot. `cf[-1] = 250` sets the **last** slot.

Years 0 to 5 are **six** dates, not five. How many slots does `np.zeros(years)` give?

You should get **25.7%** for the first version and **20.1%** for the second.

**Your prediction:** which version puts the exit in year 5? Which in year 4?

In [ ]:
import numpy as np

years = 5

# The AI's code
cf = np.zeros(years)       # how many slots?
cf[0] = -100               # equity in
cf[-1] = 250               # exit
print(f"AI's code: {npf.irr(cf):.1%}   ({len(cf)} slots)")

# The fix
cf = np.zeros(years + 1)   # how many slots?
cf[0] = -100
cf[-1] = 250
print(f"Fixed:     {npf.irr(cf):.1%}   ({len(cf)} slots)")

print(f"Put the answer back in: 100 grown for {years} years = {100 * (1 + npf.irr(cf)) ** years:.0f}")

## Cell 8 - A bad bracket

**This cell fails on purpose.** We ask `brentq` to search between 10% and 50%.

Read the message: **last line first, then the arrow.**

1. The last line says the type of error and what went wrong.
2. The arrow `---->` shows the line that failed.
3. Ask: which value is not what I think it is?

You saw the NPV at 10% in Cell 1. What is the NPV at 50%?

**Your prediction:** what will the last line say? What does `brentq` need to see at the two ends of the bracket?

(Stuck? Ask Gemini to *explain* the error. Then fix it yourself.)

In [ ]:
brentq(npv_at, 0.10, 0.50)

Fix it yourself: choose a bracket where the NPV has **opposite signs** at the two ends. You already have two rates from Cell 1.

In [ ]:
# YOUR CODE HERE: call brentq again with a bracket that has the IRR inside. Save the result in a variable called irr_fixed

In [ ]:
print(f"IRR with a good bracket: {irr_fixed:.4%}")

if abs(irr_fixed - irr) > 1e-9:
    raise ValueError("This should be the same IRR as in Cell 4")

## Cell 9 - The mine: two zeros

Now a new project, from the Lecture 4 exam: a **mine**. You pay 17,000 to open it, earn 16,000 a year for four years, then pay 52,000 in year 5 to close it and clean up. The hurdle rate is 8%.

The cash flows change sign twice: minus, plus, minus. `npf.irr` says **6.77%**, below 8%. But the NPV at 8% is **+604**.

The loop tries 161 rates from 0% to 80% (`np.linspace(0, 0.8, 161)` gives 161 evenly spaced values) and stores each NPV with `.append`. Then `plt.plot` draws the curve.

You should get two IRRs: **6.77%** and **65.4%**.

**Your prediction:** how many times will the curve cross zero? Is the NPV positive or negative between the crossings?

In [ ]:
import matplotlib.pyplot as plt

mine = [-17000, 16000, 16000, 16000, 16000, -52000]   # year 0 to 5


def mine_npv(r):
    return mine[0] + discount(mine[1:], r)


rates = np.linspace(0, 0.80, 161)
npvs = []
for r in rates:
    npvs.append(mine_npv(r))

plt.plot(rates * 100, npvs)
plt.axhline(0, color="grey")
plt.xlabel("Discount rate (%)")
plt.ylabel("NPV")
plt.title("NPV of the mine: two zeros")
plt.show()

print(f"NPV at 8%: {mine_npv(0.08):+,.0f}")
print(f"First IRR:  {brentq(mine_npv, 0, 0.2):.2%}")     # bracket 1: 0% to 20%
print(f"Second IRR: {brentq(mine_npv, 0.2, 1):.1%}")     # bracket 2: 20% to 100%

## Cell 10 - MIRR: one answer

The **MIRR** (modified IRR) avoids the problem. It assumes the cash you receive is **reinvested** at one rate, and the cash you pay is **financed** at one rate. Then there is one answer.

`npf.mirr(mine, 0.08, 0.08)`: the cash flows, the financing rate (8%) and the reinvestment rate (8%, the cost of capital).

You should get **8.25%**.

**Your prediction:** is it above or below the 8% hurdle? Does that agree with the NPV (+604)?

In [ ]:
print(f"MIRR at 8%: {npf.mirr(mine, 0.08, 0.08):.2%}")
print(f"NPV at 8%:  {mine_npv(0.08):+,.0f}")

## What you learned

**Python**
- **Import from a module**: `from scipy.optimize import brentq`. **Package install**: `!pip install numpy-financial`, then `import numpy_financial as npf`.
- **Slice**: `cash_flows[1:]`, the list without the first item. **Function**: `npv_at(r)`, one input, the unknown.
- **Function as an input**: `brentq(npv_at, 0, 1)`. No brackets after `npv_at`: you hand over the question, the solver asks it many times.
- **Bracket**: the `0, 1`, the two ends of the search. It needs a sign change inside.
- **Bisection**: a `for` loop that halves the interval (`low`, `high`, `mid`) 30 times.
- **Guard**: `if ...: raise ValueError("...")`. **Error type**: `ValueError: f(a) and f(b) must have different signs`, a bad bracket.
- **NumPy**: `np.zeros(years + 1)`, `np.linspace(0, 0.8, 161)`. **Chart**: `plt.plot`. **Financial functions**: `npf.irr`, `npf.mirr`.

**Finance**
- The IRR is the rate where the NPV is zero. Goal Seek is: try, check, adjust. Project: IRR **6.74%**, NPV +232.49 at 5% and -395.81 at 10%.
- Put the answer back in: the NPV at the IRR is 0.
- Session 0 slide 10: `np.zeros(years)` gave 25.7%, `np.zeros(years + 1)` gave 20.1%. Count the dates, years 0 to 5 are six.
- A solver finds one zero and says nothing about the others. The mine has two IRRs, **6.77%** and **65.4%**. Plot the NPV. Use the NPV or the MIRR (**8.25%**) to decide.

Keep two pages open from now on:
- [`python-card.md`](https://github.com/niparis/gem-python-finance/blob/main/python-card.md) - what each piece of Python means
- [`checks-card.md`](https://github.com/niparis/gem-python-finance/blob/main/checks-card.md) - how to know a number is right